# A real GuideLLM run from the Aurora Workbench

Select **Aurora Inference Demo** in the kernel picker. Configure the endpoint, served model, and authentication mode in the first code cell. The prepared default is Aurora Qwen through the private Gateway; it uses the Workbench identity, native model authorization, and the two existing Qwen backends.

The first run requests at most **3 completions**, **0.1 requests/second**, **concurrency 1**, and **64 output tokens**. A separate watchdog covers setup and inference. No background traffic remains when the cell finishes. Interrupting the cell kills its process group; the watchdog also stops it if the notebook kernel exits. A request already accepted by a server can finish after the client disconnects.

This is a short functional observation using fictional Aurora Supply prompts, not a model-quality evaluation, a capacity test, or proof that routing/cache behavior caused a speedup.


In [ ]:
# Change these values to select an OpenAI-compatible model endpoint.
BASE_URL = "http://showroom-inference-maas-gateway-class.ai-showroom.svc.cluster.local:8080/ai-showroom/aurora-qwen-4b/v1"
MODEL_ID = "aurora-qwen-4b"  # Exact served ID returned by this endpoint's /models API.
AUTH_MODE = "service_account"  # "service_account" or "api_key"

# GuideLLM needs the tokenizer that matches the served model.
TOKENIZER_ID = "Qwen/Qwen3-4B-Instruct-2507"
TOKENIZER_REVISION = "cdbee75f17c01a7cc42f958dc650907174af0554"
TOKENIZER_FOR_MODEL = "aurora-qwen-4b"  # Explicit binding; update only after checking the model's tokenizer.

from showroom_workbench import validate_config, prompt_api_key
CONFIG = validate_config({"base_url": BASE_URL, "model_id": MODEL_ID, "auth_mode": AUTH_MODE})
API_KEY = prompt_api_key(CONFIG) if AUTH_MODE == "api_key" else None
print("Configuration accepted for", MODEL_ID, "using", AUTH_MODE)
# Never print API_KEY, connection(), or a raw credential.


## Authentication and tokenizer selection

**Service account:** the automatic Workbench credential is accepted only for the exact private Gateway on port 8080 and a matching `/ai-showroom/{model}/v1` path. Existing routing and RBAC authorize only the prepared Qwen model; changing the model name does not grant access. A failed model preflight stops before generation.

**API key:** choose `AUTH_MODE = "api_key"` and paste an HTTPS OpenAI-compatible base URL ending in `/v1`. Enter the key only in the masked prompt. It is bound to that exact endpoint; changing the endpoint requires entering a new key. Redirects and environment proxies are disabled, and TLS verification remains enabled. No Workbench service-account token is sent to an external endpoint.

The Qwen preset uses the prepared five-file hash-verified tokenizer. For another model, explicitly set its public Hugging Face `TOKENIZER_ID`, immutable 40-character commit `TOKENIZER_REVISION`, and `TOKENIZER_FOR_MODEL`. The next cell fetches only allowlisted tokenizer/configuration files, never weights or Python code. A stale Qwen binding is refused. You are responsible for matching a remote model to its tokenizer: `/models` proves that the served ID is available, not which weights or tokenizer the server uses. A frontier endpoint without a known compatible public tokenizer can use notebook 07's manual client instead of claiming valid GuideLLM token measurements.


In [ ]:
from pathlib import Path
import importlib.metadata
import sys
from guidellm_workbench import run, display_results, prepare_tokenizer

if TOKENIZER_FOR_MODEL != MODEL_ID:
    raise ValueError("Choose and explicitly bind the matching tokenizer to MODEL_ID before running.")
tokenizer_path = prepare_tokenizer(TOKENIZER_ID, TOKENIZER_REVISION, TOKENIZER_FOR_MODEL)
print("Kernel:", sys.executable)
print("GuideLLM:", importlib.metadata.version("guidellm"))
print("Tokenizer files verified; no inference started.")


## Run the short test

Run the next cell once. The child first verifies that `/models` lists the exact selected model. No generation starts if access is denied, the route is missing, or that model is absent.

For service-account mode, the rotating credential is read only inside the child. For API-key mode, the endpoint-bound key crosses to the child through stdin only. Neither mode places a credential in process arguments, environment variables, reports, or notebook output. The prepared same-namespace Gateway hop uses HTTP with native authentication and scoped network access; API-key endpoints require verified HTTPS.

The fixed Aurora prompts share a prefix. This supplies useful context when viewing cache charts, but the short run does not isolate cache effects.


In [ ]:
result_dir = run(
    seconds=30, rate=0.1, concurrency=1, output_tokens=64,
    config=CONFIG, api_key=API_KEY, tokenizer=tokenizer_path,
    tokenizer_id=TOKENIZER_ID, tokenizer_revision=TOKENIZER_REVISION,
    tokenizer_for_model=TOKENIZER_FOR_MODEL,
)


## Read the actual result

`benchmarks.json` is the GuideLLM report, sanitized in memory before its first disk write. `summary.json`, `summary.csv`, and `requests.csv` contain derived numerical views. `benchmarks.html` is a standalone summary of those same measurements with local, embedded charts; it does not download an external report template.

Reports persist on the Workbench PVC under `/opt/app-root/src/aurora-results/guidellm/`. Each run receives its own directory. No existing report is overwritten. Failed, timed-out, or interrupted runs retain their status rather than being shown as successful.

**Streaming TTFT** is shown only for successful requests with multiple timed content events and an observed delivery interval. It is client-observed time to the first content token, including Gateway effects. If the events are unavailable or appear buffered, the notebook labels TTFT **UNAVAILABLE_OR_BUFFERED**. A missing value is not zero. Three requests are too few for a reliable percentile or capacity conclusion.

Saved settings identify the selected model, authentication mode, and tokenizer revision. The endpoint is redacted. Remote model weights are not attested by the models API. `preflight.json` retains the safe discovery result.


In [ ]:
display_results(result_dir)


In [ ]:
# Only sanitized reports and fixed run settings are listed here.
for path in sorted(Path(result_dir).iterdir()):
    if path.is_file():
        print(path.name, path.stat().st_size, "bytes")


## Connect the notebook to the existing native dashboards

These native chart instructions describe the prepared private Qwen default. For another model, select its actual OpenShift AI project/model. An external API is not automatically scraped into the local dashboards.

Open **Observe & monitor → Dashboard** in OpenShift AI. For each tab, recheck **Project = ai-showroom** and **Model = aurora-qwen-4b**; changing tabs can reset a selection. Set a time range that includes the UTC timestamp in `settings.json`.

- **LLM Traffic:** look for requests and token throughput near the run window.
- **LLM Performance:** inspect TTFT, end-to-end latency, and prefix-cache hit behavior. The native inter-token-latency panel is unavailable with this installed metric mapping; do not interpret its fallback as a measured zero.
- **LLM Utilization:** inspect the two Qwen backends, running/waiting requests, and the collected GPU series.

A three-request burst can fall between scrape intervals or be diluted by the chart window. Native charts also include other Qwen clients. Compare the notebook's client timestamps and totals with those limits in mind; do not attribute every chart change to this notebook. The **Usage** dashboard records MaaS subscriptions; this private, Kubernetes-authenticated Qwen path is not the separate `showroom-load` Llama MaaS subscription.

See [the native dashboard guide](https://weslleyrosalem.com/rhoai-showroom/operations/native-dashboards/) for exact panel semantics and known query limits.


## Optional bounded test drive

After inspecting the first result, you may deliberately repeat a short run with one changed client parameter. The helper enforces at most **120 seconds of measurement**, **180 seconds total wall time**, **0.1 requests/second**, **concurrency 2**, and **128 output tokens**. Actual completions can finish before the requested duration because the request-count limit also applies. Do not run multiple notebook kernels or cells concurrently to bypass these bounds.

The following cell defines settings only; it does not start another run. Run it only after changing it into an explicit `run(...)` call during the test drive. Keep the first report for comparison, and describe any difference as an observation on a shared system.


In [ ]:
next_test = dict(seconds=30, rate=0.1, concurrency=1, output_tokens=96)
print("Optional settings only; no traffic started:", next_test)
# next_result = run(**next_test, config=CONFIG, api_key=API_KEY,
#                   tokenizer=tokenizer_path, tokenizer_id=TOKENIZER_ID,
#                   tokenizer_revision=TOKENIZER_REVISION,
#                   tokenizer_for_model=TOKENIZER_FOR_MODEL)
# display_results(next_result)


## Reproduce the environment

The prepared persistent virtual environment is `/opt/app-root/src/.venvs/aurora-inference`. Its kernel is **Aurora Inference Demo**. The verified installation uses `guidellm==0.7.4`, `torch==2.13.0+cpu`, `transformers==5.10.1`, `datasets==4.4.1`, `pydantic==2.12.5`, `httpx==0.28.1`, `numpy==2.3.5`, and `matplotlib==3.11.0`, plus the Workbench's pinned Jupyter kernel and pandas packages. The CPU wheel uses the official PyTorch CPU index; the remaining packages use the explicit official PyPI index, because the default mirror can lag.

Only tokenizer files are copied from the already deployed Qwen revision into `/opt/app-root/src/.cache/aurora-qwen-tokenizer`; no model weights or private credentials are copied. Tokenizer loading is offline and `trust_remote_code=False`.

Source APIs: [GuideLLM 0.7.4](https://github.com/vllm-project/guidellm/tree/v0.7.4), [benchmark entry point](https://github.com/vllm-project/guidellm/blob/v0.7.4/src/guidellm/benchmark/entrypoints.py), and [streaming request metrics](https://github.com/vllm-project/guidellm/blob/v0.7.4/src/guidellm/schemas/request_stats.py).
